In [1]:
import pandas as pd

sheets = pd.read_excel("../data/raw/online_retail_II.xlsx", sheet_name=None, dtype={"Invoice": str, "StockCode": str, "Description": str ,"Customer ID": str})

print(type(sheets))          # <class 'dict'>
print(sheets.keys())         # dict_keys(['2023', '2024'])

for name, frame in sheets.items():
    print(name, frame.shape)

<class 'dict'>
dict_keys(['Year 2009-2010', 'Year 2010-2011'])
Year 2009-2010 (525461, 8)
Year 2010-2011 (541910, 8)


In [2]:
from pathlib import Path
import numpy as np

# --- Paths: find the project root, so it works from notebooks/ or the root ---
def project_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / ".git").exists():
            return p
    raise FileNotFoundError("No .git folder found above " + str(start))

print(project_root)

<function project_root at 0x11f7522a0>


In [3]:
ROOT = project_root()
PROCESSED = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
PROCESSED.mkdir(parents=True, exist_ok=True)   # create if missing, no error if present
REPORTS.mkdir(parents=True, exist_ok=True)

In [4]:
# Stack vertically and reset index
combined_df = pd.concat([sheets['Year 2009-2010'], sheets['Year 2010-2011']], ignore_index=True)
print(combined_df.head)
print(combined_df.tail)

<bound method NDFrame.head of         Invoice StockCode                          Description  Quantity  \
0        489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1        489434    79323P                   PINK CHERRY LIGHTS        12   
2        489434    79323W                  WHITE CHERRY LIGHTS        12   
3        489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4        489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   
...         ...       ...                                  ...       ...   
1067366  581587     22899         CHILDREN'S APRON DOLLY GIRL          6   
1067367  581587     23254        CHILDRENS CUTLERY DOLLY GIRL          4   
1067368  581587     23255      CHILDRENS CUTLERY CIRCUS PARADE         4   
1067369  581587     22138        BAKING SET 9 PIECE RETROSPOT          3   
1067370  581587      POST                              POSTAGE         1   

                InvoiceDate  Price Customer ID         Co

In [5]:
print(sheets['Year 2009-2010'].columns.equals((sheets['Year 2010-2011'].columns)))

True


In [6]:
print(sheets['Year 2009-2010'].head(5))
print(sheets['Year 2010-2011'].head(5))

  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

          InvoiceDate  Price Customer ID         Country  
0 2009-12-01 07:45:00   6.95       13085  United Kingdom  
1 2009-12-01 07:45:00   6.75       13085  United Kingdom  
2 2009-12-01 07:45:00   6.75       13085  United Kingdom  
3 2009-12-01 07:45:00   2.10       13085  United Kingdom  
4 2009-12-01 07:45:00   1.25       13085  United Kingdom  
  Invoice StockCode                          Description  Quantity  \
0  536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1  536365     71053                  WHITE METAL LANTERN         6   
2  536365    84

In [7]:
first_sheet_daterange = sheets["Year 2009-2010"]["InvoiceDate"].max() - sheets['Year 2009-2010']['InvoiceDate'].min()
print(first_sheet_daterange)
second_sheet_daterange = sheets["Year 2010-2011"]["InvoiceDate"].max() - sheets['Year 2010-2011']['InvoiceDate'].min()
print(second_sheet_daterange)
#sheets['Year 2009-2010'].columns.tolist()
# sheets['Year 2009-2010']['Description'] 
# print(sheets["Year 2009-2010"].dtypes)
# print(sheets["Year 2010-2011"].dtypes)
print(combined_df.dtypes)

373 days 12:16:00
373 days 04:24:00
Invoice                   str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID               str
Country                   str
dtype: object


In [8]:
# toy.dtypes                 # dtype
# toy.notna().sum()          # non_null  (toy.count() does the same)
# toy.isna().sum()           # missing
# toy.isna().mean() * 100    # missing_%  (mean of True/False = fraction True)
# toy.nunique()              # n_unique, excludes NaN by default

def df_summary(df):
    print("datatypes: ----------------\n", df.dtypes)
    print("row count:---------------- \n", df.shape[0])
    print("Not NA: -----------------\n", df.notna().sum())
    print("NA count ----------------\n", df.isna().sum())
    print("missing_%: --------------\n", df.isna().mean() * 100)
    print("Unique columns ----------------\n", df.nunique())

#df_summary(sheets["Year 2009-2010"])
# df_summary(sheets["Year 2010-2011"])
df_summary(combined_df)




datatypes: ----------------
 Invoice                   str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID               str
Country                   str
dtype: object
row count:---------------- 
 1067371
Not NA: -----------------
 Invoice        1067371
StockCode      1067371
Description    1062989
Quantity       1067371
InvoiceDate    1067371
Price          1067371
Customer ID     824364
Country        1067371
dtype: int64
NA count ----------------
 Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64
missing_%: --------------
 Invoice         0.000000
StockCode       0.000000
Description     0.410541
Quantity        0.000000
InvoiceDate     0.000000
Price           0.000000
Customer ID    22.766873
Country         0.000000
dtype: float6

In [11]:
ID_COLS = ["Invoice", "StockCode", "Description", "Quantity",
           "InvoiceDate", "Price", "Customer ID", "Country"]   # the original 8 columns

PRODUCT_CODE = r"^\d{5}" 

def apply_rule(df, keep_fn, rule, log):
    """Keep rows where keep_fn(df) is True; record what was removed."""
    before = len(df)
    out = df[keep_fn(df)].copy()
    log.append({
        "rule": rule,
        "rows_before": before,
        "removed": before - len(out),
        "pct_removed": round(100 * (before - len(out)) / before, 2),
        "rows_after": len(out),
    })
    return out    

def clean_data(raw):
    log = []
    df = raw.copy()
    df["StockCode"] = df["StockCode"].str.strip().str.upper()    # normalise, no removal
    df = apply_rule(df, lambda d: ~d.duplicated(subset=ID_COLS, keep="first"),
                    "Exact duplicate rows", log)
    df = apply_rule(df, lambda d: ~d["Invoice"].str.startswith("C", na=False), "Cancellations (Invoice starts with C)", log)
    df = apply_rule(df, lambda d: d["Customer ID"].notna(), "Missing Customer ID", log)
    df = apply_rule(df, lambda d: d["Quantity"] > 0, "Quantity <= 0", log)
    df = apply_rule(df, lambda d: d["Price"] > 0, "Price <= 0", log)  
    df = apply_rule(df, lambda d: d["StockCode"].str.match(PRODUCT_CODE, na=False),
                    "Non-product codes", log)

    return df, pd.DataFrame(log)

clean_df, log_df = clean_data(combined_df)
print(log_df)

                                    rule  rows_before  removed  pct_removed  \
0                   Exact duplicate rows      1067371    34335         3.22   
1  Cancellations (Invoice starts with C)      1033036    19104         1.85   
2                    Missing Customer ID      1013932   234437        23.12   
3                          Quantity <= 0       779495        0         0.00   
4                             Price <= 0       779495       70         0.01   
5                      Non-product codes       779425     2848         0.37   

   rows_after  
0     1033036  
1     1013932  
2      779495  
3      779495  
4      779425  
5      776577  


In [16]:
# --- Save ---
clean_df.to_parquet(PROCESSED / "transactions_clean.parquet", index=False)
log_df.to_csv(REPORTS / "cleaning_log.csv", index=False)

In [15]:
# sc = combined_df["StockCode"].str.strip().str.upper()
# non_prod = combined_df[~sc.str.match(PRODUCT_CODE, na=False)].assign(StockCode=sc)
# non_prod_summary = (non_prod.groupby("StockCode")["Description"]
#                             .agg(n_rows="size", example="first")
#                             .sort_values("n_rows", ascending=False))

# print(len(non_prod_summary), "distinct non-product codes")
# print(non_prod_summary.to_string())

In [32]:
def split_data(clean_df, test_months=2):
    """Time-based split. Returns train rows, test (customer, product) pairs, and a report."""
    report = []
    def note(metric, value):
        report.append({"metric": metric, "value": value})

    # --- 1. Time split ---
    cutoff = clean_df["InvoiceDate"].max() - pd.DateOffset(months=test_months)
    train_df = clean_df[clean_df["InvoiceDate"] <= cutoff]
    test_df = clean_df[clean_df["InvoiceDate"] > cutoff]
    note("cutoff (train <= cutoff < test)", str(cutoff))
    note("train rows", len(train_df))
    note("test rows (raw)", len(test_df))
    note("train customers", train_df["Customer ID"].nunique())
    note("train products", train_df["StockCode"].nunique())
    note("test customers (raw)", test_df["Customer ID"].nunique())

    # --- 2. Keep only test customers with training history ---
    test_f = test_df[test_df["Customer ID"].isin(train_df["Customer ID"])]
    note("test customers with train history", test_f["Customer ID"].nunique())
    note("test customers dropped (new, cold start)",
         test_df["Customer ID"].nunique() - test_f["Customer ID"].nunique())

    # --- 3. Keep only products the customer never bought in train (anti-join) ---
    train_pairs = train_df[["Customer ID", "StockCode"]].drop_duplicates()
    test_f = test_f.merge(train_pairs, on=["Customer ID", "StockCode"],
                          how="left", indicator=True)
    test_f = test_f[test_f["_merge"] == "left_only"].drop(columns=["_merge"])

    # --- 4. Keep only products seen in train ---
    test_f = test_f[test_f["StockCode"].isin(train_df["StockCode"])]

    # --- 5. Unique held-out pairs ---
    test_pairs = test_f[["Customer ID", "StockCode"]].drop_duplicates().reset_index(drop=True)
    n_eval = test_pairs["Customer ID"].nunique()
    note("held-out (customer, product) pairs", len(test_pairs))
    note("evaluation customers (>= 1 held-out pair)", n_eval)
    note("avg new products per evaluation customer", round(len(test_pairs) / n_eval, 2))

    # --- 6. R2 leak checks ---
    assert train_df["InvoiceDate"].max() <= cutoff < test_df["InvoiceDate"].min(), \
        "Time overlap between train and test!"
    overlap = test_pairs.merge(train_pairs, on=["Customer ID", "StockCode"], how="inner")
    assert len(overlap) == 0, f"{len(overlap)} test pairs were already bought in train!"
    assert test_pairs["Customer ID"].isin(train_df["Customer ID"]).all(), \
        "Test customer with no training history!"

    return train_df, test_pairs, pd.DataFrame(report)


train_df, test_pairs, split_report = split_data(clean_df)
print(split_report.to_string(index=False))

# --- Save ---
train_df.to_parquet(PROCESSED / "train.parquet", index=False)
test_pairs.to_parquet(PROCESSED / "test_pairs.parquet", index=False)
split_report.to_csv(REPORTS / "split_report.csv", index=False)

# --- Round-trip check ---
assert pd.read_parquet(PROCESSED / "test_pairs.parquet").shape == test_pairs.shape

                                   metric               value
          cutoff (train <= cutoff < test) 2011-10-09 12:50:00
                               train rows              660194
                          test rows (raw)              116383
                          train customers                5474
                           train products                4529
                     test customers (raw)                2420
        test customers with train history                2042
 test customers dropped (new, cold start)                 378
       held-out (customer, product) pairs               45753
evaluation customers (>= 1 held-out pair)                1916
 avg new products per evaluation customer               23.88


In [ ]:
import numpy as np
import scipy.sparse as sp

########################### DE4 ##############################
def build_matrix(train_df, test_pairs, min_customers=10, purchase_value="log1p"):
    """Build the customer x product matrix from training data.

    Returns the matrix, the customer list (row order), the product list
    (column order) and the test pairs that fit the matrix.
    """

    # Step 1: Add up how much each customer bought of each product
    pairs = train_df.groupby(["Customer ID", "StockCode"])["Quantity"].sum().reset_index()
    print("Customer-product pairs:", len(pairs))

    # Step 2: Product filter
    customers_per_product = pairs.groupby("StockCode")["Customer ID"].nunique()
    popular_products = customers_per_product[customers_per_product >= min_customers].index #selecting products with more than minimum customers
    print("Products before filter:", len(customers_per_product))
    print("Products kept:", len(popular_products))
    pairs = pairs[pairs["StockCode"].isin(popular_products)]

    # Step 3: Row and column numbers
    customer_list = sorted(pairs["Customer ID"].unique())                        # sorted the unique customer ids
    product_list = sorted(pairs["StockCode"].unique())                           # sorted the unique product ids
    customer_to_row = {customer: i for i, customer in enumerate(customer_list)}  # created dict for customer and its position
    product_to_col = {product: j for j, product in enumerate(product_list)}      # created dict for products and its position
    print("Customers in matrix:", len(customer_list))
    print("Train customers lost:", train_df["Customer ID"].nunique() - len(customer_list))

    # Step 4: Purchase value selection
    if purchase_value == "log1p":
        values = np.log1p(pairs["Quantity"]) # Apply log1p transformation to the quantities in our pair data.
    elif purchase_value == "binary":
        values = np.ones(len(pairs))
    elif purchase_value == "count":
        values = pairs["Quantity"]
    else:
        raise ValueError(f"Unknown purchase_value: {purchase_value}")
    values = np.asarray(values, dtype=float)

    # Step 5: Sparse matrix
    rows = pairs["Customer ID"].map(customer_to_row).to_numpy() #.map() swaps each ID in pairs for its number using the dictionary (customer_to_row)
    cols = pairs["StockCode"].map(product_to_col).to_numpy() #.map() swaps each ID in pairs for its number using the dictionary (product_to_col)
    A = sp.csr_matrix((values, (rows, cols)), shape=(len(customer_list), len(product_list))) 
    print("Matrix shape:", A.shape)
    assert A.nnz == len(pairs)

    # Step 6: Test pairs that fit the matrix
    keep = (test_pairs["Customer ID"].isin(customer_list)
            & test_pairs["StockCode"].isin(product_list))
    test_pairs_matrix = test_pairs[keep].reset_index(drop=True)
    print("Test pairs kept:", len(test_pairs_matrix), "of", len(test_pairs))
    print("Evaluation customers:", test_pairs_matrix["Customer ID"].nunique())

    return A, customer_list, product_list, test_pairs_matrix

In [42]:
A, customer_list, product_list, test_pairs_matrix = build_matrix(train_df, test_pairs)

sp.save_npz(PROCESSED / "A_train.npz", A)
pd.DataFrame({"Customer ID": customer_list}).to_parquet(PROCESSED / "customer_index.parquet", index=False)
pd.DataFrame({"StockCode": product_list}).to_parquet(PROCESSED / "product_index.parquet", index=False)
test_pairs_matrix.to_parquet(PROCESSED / "test_pairs_matrix.parquet", index=False)

Customer-product pairs: 417719
Products before filter: 4529
Products kept: 3687
Customers in matrix: 5473
Train customers lost: 1
Matrix shape: (5473, 3687)
Test pairs kept: 44727 of 45753
Evaluation customers: 1913


In [60]:
########################### DE5 ##############################

def create_matrix_report(A, customer_list, product_list):
    # print("Matrix shape:", A.shape)
    # print("Number of non-zero elements:", A.nnz)
    # print("Sparsity:", 1 - A.nnz / (A.shape[0] * A.shape[1]))

    report = []
    def note(metric, value):
        report.append({"metric": metric, "value": value})

    dense_mb = round((A.shape[0] * A.shape[1]*8)/1000000, 2)
    sparse_bytes = A.data.nbytes + A.indices.nbytes + A.indptr.nbytes
    sparse_mb = round(sparse_bytes / 1000000, 2)
    factors_k50_mb = round(((A.shape[0] + A.shape[1])*50*8)/1000000, 2)

    note("Matrix shape", A.shape)
    note("Number of non-zero elements", A.nnz)
    note("Sparsity %", (1 - A.nnz / (A.shape[0] * A.shape[1])) * 100)
    note("dense_memory in MBs", dense_mb)
    note("sparse_memory in MBs", sparse_mb)
    note("sparse_memory is N times smaller than dense_memory", round(dense_mb/sparse_mb, 1))
    note("compressed_memory in MBs with k = 50", factors_k50_mb)

    return pd.DataFrame(report)

matrix_report = create_matrix_report(A, customer_list, product_list)
print(matrix_report)
matrix_report.to_csv(REPORTS / "matrix_report.csv", index=False)

                                              metric         value
0                                       Matrix shape  (5473, 3687)
1                        Number of non-zero elements        413927
2                                         Sparsity %     97.948719
3                                dense_memory in MBs        161.43
4                               sparse_memory in MBs          4.99
5  sparse_memory is N times smaller than dense_me...          32.4
6               compressed_memory in MBs with k = 50          3.66
